In [9]:
import os, math, json, random
import torch
import torch.nn as nn
from torch.utils.data import random_split
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINEConv, global_add_pool
import torch.nn.functional as F
from itertools import product
from torch_geometric.data import Batch
from typing import Tuple, Dict, Any

In [10]:
# ==== your existing constants & feature engineering (unchanged) ====

NODE_FEATURE_DIM = 10
EDGE_FEATURE_DIM = 4

OP_TYPES = ['kn_input_op', 
            'kn_output_op', 
            'kn_add_op', 
            'kn_div_op',
            'kn_exp_op', 
            'kn_matmul_op', 
            'kn_mul_op',
            'kn_pow_op', 
            'kn_reduction_2_op', 
            'kn_sqrt_op']

def one_hot_op_type(op_type):
    out = torch.zeros(NODE_FEATURE_DIM, dtype=torch.float)
    out[OP_TYPES.index(op_type)] = 1.0
    return out

def compute_flops(op_type, input_tensors):
    if op_type == 'kn_matmul_op':
        M = input_tensors[0][-2]
        K = input_tensors[0][-1]
        N = input_tensors[1][-1]
        return 2 * M * N * K
    elif op_type in ['kn_add_op', 
                     'kn_div_op', 
                     'kn_exp_op', 
                     'kn_mul_op', 
                     'kn_pow_op', 
                     'kn_sqrt_op', 
                     'kn_reduction_2_op']:
        num_elements = 1
        for dim in input_tensors[0]:
            num_elements *= dim
        return num_elements
    elif op_type in ['kn_input_op', 'kn_output_op']:
        return 0
    else:
        raise ValueError(f"Unknown op type: {op_type}")

def get_node_features(node):
    # one-hot op type
    h_op_type = one_hot_op_type(node['op_type'])
    
    # in tensor dimensions
    in_tensors = []
    h_in_dims = torch.zeros(NODE_FEATURE_DIM, dtype=torch.float)
    for i, t in enumerate(node['input_tensors']):
        in_tensors.append(t['dim'])
        for j, dim in enumerate(t['dim']):
            h_in_dims[j + (i * 4)] = dim
    
    # out tensor dimensions
    out_tensors = []
    h_out_dims = torch.zeros(NODE_FEATURE_DIM, dtype=torch.float)
    for i, t in enumerate(node['output_tensors']):
        out_tensors.append(t['dim'])
        for j, dim in enumerate(t['dim']):
            h_out_dims[j + (i * 4)] = dim
    
    # in tensor sizes
    h_in_size = torch.zeros(NODE_FEATURE_DIM, dtype=torch.float)
    for i, t in enumerate(in_tensors):
        h_in_size[i] = math.prod(t)
    
    # out tensor sizes
    h_out_size = torch.zeros(NODE_FEATURE_DIM, dtype=torch.float)
    for i, t in enumerate(out_tensors):
        h_out_size[i] = math.prod(t)

    h_out_size = torch.zeros(NODE_FEATURE_DIM, dtype=torch.float)
    for i, t in enumerate(out_tensors):
        h_out_size[i] = math.prod(t)

    # computation cost in FLOPs
    flops = compute_flops(node['op_type'], in_tensors)
    h_flops = torch.zeros(NODE_FEATURE_DIM, dtype=torch.float)
    h_flops[0] = flops

    return torch.vstack([h_op_type, h_in_dims, h_out_dims, h_in_size, h_out_size, h_flops])

def get_edge_features(tensor):
    h_edge = torch.zeros(EDGE_FEATURE_DIM, dtype=torch.float)
    for i, dim in enumerate(tensor['dim']):
        h_edge[i] = dim
    
    h_edge_size = torch.zeros(EDGE_FEATURE_DIM, dtype=torch.float)
    h_edge_size[0] = math.prod(tensor['dim'])

    return torch.vstack([h_edge, h_edge_size])

# ==== new: dataset that yields ALL pairs ====

class PairSubgraphDataset(Dataset):
    """
    Emits (data_a, data_b, gt) per index, where:
      - data_a, data_b: PyG Data objects for two different subgraphs, with NO `y` field.
      - gt: torch.bool tensor shaped [1], True iff exec_time(a) > exec_time(b).

    By default, produces all ORDERED pairs (i, j) with i != j, so len = N * (N - 1).
    Set ordered=False to get unordered combinations (i < j), len = N * (N - 1) // 2.
    """
    def __init__(self, root, ordered: bool = True):
        super().__init__(root)
        self.root = root
        self.ordered = ordered

        json_files = [f for f in os.listdir(root) if f.endswith('.json') and f.startswith('original_')]
        hash_to_time = json.load(open(os.path.join(root, 'performance.json'), 'r'))

        self.subgraph_files = []
        self.execution_times = []
        for json_file in json_files:
            h = json_file[len('original_'):-len('.json')]
            if h in hash_to_time:
                self.subgraph_files.append(json_file)
                self.execution_times.append(hash_to_time[h])

        assert len(self.subgraph_files) == len(self.execution_times)
        self.N = len(self.subgraph_files)

        # Precompute index pairs for simplicity & speed.
        self.pairs = []
        if self.ordered:
            # All permutations i != j
            for i in range(self.N):
                for j in range(self.N):
                    if i != j:
                        self.pairs.append((i, j))
        else:
            # All combinations i < j
            for i in range(self.N):
                for j in range(i + 1, self.N):
                    self.pairs.append((i, j))

    def len(self):
        return len(self.pairs)

    def _load_graph(self, idx: int) -> Data:
        """Load a single subgraph as a PyG Data WITHOUT a `y` field."""
        json_path = os.path.join(self.root, self.subgraph_files[idx])
        json_graph = json.load(open(json_path, 'r'))

        nodes = []
        edge_index = []
        edge_attr = []
        producer_of = {}

        for node_idx, node in enumerate(json_graph):
            nodes.append(get_node_features(node))
            for t in node['output_tensors']:
                producer_of[t['guid']] = node_idx

        for dst_idx, node in enumerate(json_graph):
            for t in node['input_tensors']:
                src_idx = producer_of[t['guid']]
                edge_index.append([src_idx, dst_idx])
                edge_attr.append(get_edge_features(t))

        return Data(
            x=torch.stack(nodes, dim=0),
            edge_index=torch.tensor(edge_index, dtype=torch.long).t().contiguous() if edge_index else torch.empty((2,0), dtype=torch.long),
            edge_attr=torch.stack(edge_attr, dim=0) if edge_attr else torch.empty((0, 2, EDGE_FEATURE_DIM), dtype=torch.float),
            # IMPORTANT: no `y` here
        )

    def get(self, idx):
        i, j = self.pairs[idx]

        data_i = self._load_graph(i)
        data_j = self._load_graph(j)

        # Boolean label: True iff time(i) > time(j)
        gt = torch.tensor([self.execution_times[i] > self.execution_times[j]], dtype=torch.bool)

        # Return as a tuple; PyG DataLoader will collate these.
        return data_i, data_j, gt

In [11]:
# -----------------------------
# Helper: flatten features
# -----------------------------
def flatten_node_x(x):
    # from [N, 6, 10] -> [N, 60] without altering feature values
    return x.view(x.size(0), -1)

def flatten_edge_attr(edge_attr):
    # from [E, 2, EDGE_FEATURE_DIM] -> [E, 2*EDGE_FEATURE_DIM]
    return edge_attr.view(edge_attr.size(0), -1)

def pair_collate(batch):
    data_a_list, data_b_list, y_list = [], [], []
    for a, b, gt in batch:
        data_a_list.append(a)
        data_b_list.append(b)
        y_list.append(gt)

    batch_a = Batch.from_data_list(data_a_list)
    batch_b = Batch.from_data_list(data_b_list)
    y = torch.stack(y_list, dim=0).float()  # [B, 1], will be 0./1. after float()
    return batch_a, batch_b, y

# -----------------------------
# GINE-based graph encoder
# -----------------------------
class GINEEncoder(nn.Module):
    def __init__(self, in_channels, edge_dim, hidden=128, out_channels=128, num_layers=3, train_eps=True, dropout=0.1):
        super().__init__()
        assert num_layers >= 1

        def make_mlp(cin, cout):
            return nn.Sequential(
                nn.Linear(cin, hidden),
                nn.ReLU(),
                nn.Linear(hidden, cout),
            )

        self.convs = nn.ModuleList()
        self.bns   = nn.ModuleList()

        # first layer
        self.convs.append(GINEConv(nn=make_mlp(in_channels, hidden), edge_dim=edge_dim, train_eps=train_eps))
        self.bns.append(nn.BatchNorm1d(hidden))

        # hidden layers
        for _ in range(num_layers - 2):
            self.convs.append(GINEConv(nn=make_mlp(hidden, hidden), edge_dim=edge_dim, train_eps=train_eps))
            self.bns.append(nn.BatchNorm1d(hidden))

        # last layer (project to out_channels)
        if num_layers > 1:
            self.convs.append(GINEConv(nn=make_mlp(hidden, out_channels), edge_dim=edge_dim, train_eps=train_eps))
            self.bns.append(nn.BatchNorm1d(out_channels))
        else:
            # if num_layers==1 the first layer already outputs hidden; align dims
            self.bns[-1] = nn.BatchNorm1d(hidden)
            out_channels = hidden

        self.out_channels = out_channels
        self.dropout = dropout

    def forward(self, data):
        # data: PyG Data with .x, .edge_index, .edge_attr, .batch
        x = flatten_node_x(data.x)                         # [N, 60] with your constants
        edge_attr = flatten_edge_attr(data.edge_attr) if data.edge_attr.numel() > 0 else data.edge_attr

        for i, conv in enumerate(self.convs):
            x = conv(x, data.edge_index, edge_attr)
            x = self.bns[i](x)
            if i < len(self.convs) - 1:
                x = F.relu(x)
                x = F.dropout(x, p=self.dropout, training=self.training)

        # Graph-level embedding
        g = global_add_pool(x, data.batch)                 # [B, out_channels]
        return g

# -----------------------------
# Pairwise comparator head
# -----------------------------
class PairwiseGINEClassifier(nn.Module):
    """
    Predicts whether exec_time(graph_a) > exec_time(graph_b).
    Forward accepts (batch_a, batch_b) or a dict with keys 'a' and 'b'.
    """
    def __init__(self, node_in_channels=6*10, edge_in_channels=2*4, hidden=128, encoder_layers=3, dropout=0.1):
        super().__init__()
        self.encoder = GINEEncoder(
            in_channels=node_in_channels,
            edge_dim=edge_in_channels,
            hidden=hidden,
            out_channels=hidden,
            num_layers=encoder_layers,
            dropout=dropout,
        )
        comp_in = 3 * hidden  # [hA, hB, |hA - hB|]
        self.clf = nn.Sequential(
            nn.Linear(comp_in, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1)  # logits
        )

    def encode(self, data):
        return self.encoder(data)

    def forward(self, batch_a, batch_b):
        """
        batch_a, batch_b: PyG Batch (batched Data) for first and second subgraphs.
        Returns logits of shape [B, 1] (use BCEWithLogitsLoss).
        """
        h_a = self.encode(batch_a)                      # [B, H]
        h_b = self.encode(batch_b)                      # [B, H]
        h = torch.cat([h_a, h_b, (h_a - h_b).abs()], dim=-1)
        logits = self.clf(h)                            # [B, 1]
        return logits

In [16]:
# -------- helpers ----------
def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

@torch.no_grad()
def evaluate_cls(model, loader, device) -> Tuple[float, float]:
    """
    Returns (val_loss, accuracy) using BCEWithLogitsLoss and 0.5 threshold.
    """
    model.eval()
    criterion = nn.BCEWithLogitsLoss()
    total_loss, total, correct = 0.0, 0, 0
    for batch_a, batch_b, y in loader:
        batch_a, batch_b, y = batch_a.to(device), batch_b.to(device), y.to(device).view(-1, 1).float()
        logits = model(batch_a, batch_b)           # [B,1]
        loss = criterion(logits, y)
        total_loss += loss.item() * y.size(0)
        preds = (logits.sigmoid() > 0.5).long()    # [B,1]
        correct += (preds == y.long()).sum().item()
        total += y.numel()
    if total == 0:
        return float("nan"), float("nan")
    return total_loss / total, correct / total

def train_one_pair(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    device,
    lr: float,
    weight_decay: float,
    max_epochs: int,
    ckpt_path: str,
    log_every: int = 10,
):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=5, verbose=False)
    criterion = nn.BCEWithLogitsLoss()

    best_val_loss = float('inf')
    best_state = None

    for epoch in range(1, max_epochs + 1):
        model.train()
        train_loss_sum, total = 0.0, 0
        for batch_a, batch_b, y in train_loader:
            batch_a, batch_b, y = batch_a.to(device), batch_b.to(device), y.to(device).view(-1, 1).float()

            opt.zero_grad()
            logits = model(batch_a, batch_b)     # [B,1]
            loss = criterion(logits, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            opt.step()

            train_loss_sum += loss.item() * y.size(0)
            total += y.numel()

        train_loss = train_loss_sum / max(1, total)
        val_loss, val_acc = evaluate_cls(model, val_loader, device)
        scheduler.step(val_loss)

        if epoch % log_every == 0 or epoch == 1:
            print(f"Epoch {epoch:03d} | Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f} | Val acc: {val_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = model.state_dict()
            torch.save(best_state, ckpt_path)

    if best_state is not None:
        model.load_state_dict(best_state)
    return best_val_loss

# -------- main grid ----------
def train_eval_grid(
    data_root: str,
    ordered: bool = True,          # PairSubgraphDataset option
    batch_size: int = 32,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    max_epochs: int = 30,
    test_ratio: float = 0.15,      # held-out test split
    val_ratio: float = 0.15,       # inner validation from train
    seed: int = 42,
    hidden_grid = (64, 128, 256),
    layers_grid = (2, 3, 4),
    dropout: float = 0.1,
    model_prefix: str = "pair_gine",
) -> Dict[str, Any]:
    """
    Grid-search over (hidden, layers) for pairwise GINE classification:
    Predict (exec_time(A) > exec_time(B)).
    """
    set_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ---- dataset + top-level split ----
    ds = PairSubgraphDataset(data_root, ordered=ordered)
    n_total = len(ds)
    assert n_total >= 3, "Need at least 3 pair samples."

    n_test = max(1, int(round(n_total * test_ratio)))
    n_trainval = n_total - n_test
    gen_main = torch.Generator().manual_seed(seed)
    trainval_ds, test_ds = random_split(ds, [n_trainval, n_test], generator=gen_main)

    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=pair_collate)

    # ---- inner split (train/val) reused for all configs ----
    n_val = max(1, int(round(len(trainval_ds) * val_ratio)))
    n_train = len(trainval_ds) - n_val
    assert n_train >= 1, "Train portion became empty; reduce val_ratio."

    gen_inner = torch.Generator().manual_seed(seed + 1)
    inner_train_subset, inner_val_subset = random_split(trainval_ds, [n_train, n_val], generator=gen_inner)

    inner_train_loader = DataLoader(inner_train_subset, batch_size=batch_size, shuffle=True,  collate_fn=pair_collate)
    inner_val_loader   = DataLoader(inner_val_subset,   batch_size=batch_size, shuffle=False, collate_fn=pair_collate)

    # ---- grid search ----
    best_cfg, best_val_acc = None, -1.0
    best_ckpt_tmp = "_pair_grid_best_tmp.pt"
    model_out = None

    for hidden, layers in product(hidden_grid, layers_grid):
        print(f"\n=== Grid config: hidden={hidden}, layers={layers} ===")
        model = PairwiseGINEClassifier(
            node_in_channels=6*10,  # (6,10) flattened
            edge_in_channels=2*4,   # (2,4) flattened
            hidden=hidden,
            encoder_layers=layers,
            dropout=dropout
        ).to(device)

        # train & keep best checkpoint for this config
        _ = train_one_pair(
            model,
            inner_train_loader,
            inner_val_loader,
            device,
            lr=lr,
            weight_decay=weight_decay,
            max_epochs=max_epochs,
            ckpt_path=best_ckpt_tmp,
        )

        # reload best for this config and evaluate on inner val
        model.load_state_dict(torch.load(best_ckpt_tmp, map_location=device))
        val_loss, val_acc = evaluate_cls(model, inner_val_loader, device)
        print(f"Config result → Val loss: {val_loss:.4f} | Val acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_cfg = {"hidden": hidden, "layers": layers}
            model_out = f"{model_prefix}_hidden{hidden}_layers{layers}.pt"
            torch.save(model.state_dict(), model_out)
            print(f"  ✔ New best config saved to {model_out}")

    assert best_cfg is not None, "Grid search evaluated no configuration."

    print(f"\nBest config: hidden={best_cfg['hidden']}, layers={best_cfg['layers']} (Val acc={best_val_acc:.4f})")

    # ---- Test the best configuration on held-out test ----
    best_model = PairwiseGINEClassifier(
        node_in_channels=6*10,
        edge_in_channels=2*4,
        hidden=best_cfg["hidden"],
        encoder_layers=best_cfg["layers"],
        dropout=dropout
    ).to(device)
    best_model.load_state_dict(torch.load(model_out, map_location=device))

    test_loss, test_acc = evaluate_cls(best_model, test_loader, device)
    print(f"\n🏁 Test results (held-out): loss={test_loss:.6f} | acc={test_acc:.4f}")

    # Also show positive rate for sanity (class balance)
    with torch.no_grad():
        pos_cnt, n_all = 0, 0
        for _, _, y in test_loader:
            n_all += y.numel()
            pos_cnt += int(y.sum().item())
    pos_rate = (pos_cnt / max(1, n_all))
    print(f"ℹ️ Test positive rate (A>B): {pos_rate:.3f} ({pos_cnt}/{n_all})")

    return {
        "best_config": best_cfg,
        "val_acc": best_val_acc,
        "test_acc": test_acc,
        "checkpoint": model_out,
    }


In [ ]:
train_eval_grid(
    data_root="/home/kitao/projects/mirage/dataset/10_16_25_cleaned",
    ordered=False,          # or False if you want unique pairs only
    batch_size=32,
    max_epochs=10,
    hidden_grid=(128, 256, 512),
    layers_grid=(4, 8, 16),
    model_prefix="models/bincomp_gine"
)


=== Grid config: hidden=128, layers=4 ===
Epoch 001 | Train loss: 0.4295 | Val loss: 0.4258 | Val acc: 0.7902
Config result → Val loss: 0.4258 | Val acc: 0.7902
  ✔ New best config saved to bincomp_gine_hidden128_layers4.pt

Best config: hidden=128, layers=4 (Val acc=0.7902)

🏁 Test results (held-out): loss=0.427998 | acc=0.7881
ℹ️ Test positive rate (A>B): 0.531 (2153/4054)
Total time: 105.97 seconds
